# Full SPARC Head-to-Head Benchmark

Runs the locked 171-galaxy NFW benchmark and reports RMS, $R^2$, AIC, and BIC for NFW and the three-parameter global boundary relation.


In [ ]:
!pip -q install numpy pandas scipy


In [ ]:
import os, shutil, json, subprocess, sys

repo_dir = '/content/Galaxy-Rotation-Curves-SPARC-Validation-Test'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone -q https://github.com/SPruynIDR/Galaxy-Rotation-Curves-SPARC-Validation-Test.git "$repo_dir"
os.chdir(repo_dir)
print('Repository ready:', repo_dir)


In [ ]:
script = 'run_full_head_to_head_benchmark.py'
if not os.path.exists(script):
    raise FileNotFoundError(
        f'{script} was not found in the repository root. '
        'Make sure the individual benchmark files were uploaded and committed.'
    )

result = subprocess.run(
    [sys.executable, script],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Benchmark failed with exit code {result.returncode}')


In [ ]:
from IPython.display import display, Markdown

with open('outputs/summary.json', 'r') as f:
    s = json.load(f)

nfw = s['nfw_result']
bnd = s['boundary_result']
cmp = s['comparison']

display(Markdown(f'''## RESULT

**Sample:** {s['sample']['galaxies']} galaxies, {s['sample']['points']} data points

| Model | RMS (km/s) | R² | AIC | BIC |
|---|---:|---:|---:|---:|
| NFW halo | {nfw['rms_km_s']:.10f} | {nfw['r2']:.10f} | {nfw['aic']:.4f} | {nfw['bic']:.4f} |
| Boundary relation | {bnd['rms_km_s']:.10f} | — | {bnd['aic']:.4f} | {bnd['bic']:.4f} |

**ΔAIC (NFW − boundary):** {cmp['delta_aic_nfw_minus_boundary']:.4f}

**ΔBIC (NFW − boundary):** {cmp['delta_bic_nfw_minus_boundary']:.4f}

**Interpretation:** AIC favors NFW. BIC favors the three-parameter global boundary relation.
'''))


In [ ]:
import pandas as pd
fits = pd.read_csv('outputs/nfw_per_galaxy_best_fits.csv')
fits.head(20)
